In [1]:
from Trust_Score_Engine import RiskScoreEngine, evaluate_auction_entry

engine = RiskScoreEngine(
    xgb_model_path="final_xgb_shill_model.json",
    xgb_metadata_path="final_model_metadata.json",
    iso_model_path="isolation_forest_model.pkl",
    iso_scaler_path="isolation_forest_scaler.pkl",
    iso_metadata_path="isolation_forest_metadata.json"
)

In [3]:
result = evaluate_auction_entry(
    engine=engine,
    user_id="user001",
    starting_price=100000,
    completed_auctions=0
)

print(result)

{'user_id': 'user001', 'user_stage': 'new', 'risk_tier': None, 'entry_allowed': True, 'risk_score': None, 'collateral': {'starting_price': 100000, 'base_collateral': 1000.0, 'collateral_multiplier': 1.0, 'required_collateral': 1000.0}, 'details': {'user_stage': 'new', 'risk_tier': None, 'final_risk_score': None, 'xgb_probability': None, 'isolation_score': None, 'xgb_threshold': None, 'iso_threshold': None, 'xgb_flag': None, 'iso_flag': None, 'xgb_score': None, 'isolation_score_normalized': None}}


In [4]:
import pandas as pd

second_time_features = pd.DataFrame([{
    "Bidder_Tendency": 0.85,
    "Bidding_Ratio": 0.75,
    "Successive_Outbidding": 10,
    "Last_Bidding": 1,
    "Auction_Bids": 20,
    "Starting_Price_Average": 500,
    "Early_Bidding": 1,
    "Winning_Ratio": 0.85,
    "Auction_Duration": 7
}])

result = evaluate_auction_entry(
    engine=engine,
    user_id="user001",
    starting_price=100000,
    completed_auctions=1,
    features=second_time_features
)

print(result)

{'user_id': 'user001', 'user_stage': 'established', 'risk_tier': 'critical', 'entry_allowed': False, 'risk_score': 99.96, 'collateral': None, 'details': {'user_stage': 'established', 'risk_tier': 'critical', 'xgb_probability': 0.9994, 'isolation_score': 0.7422, 'xgb_threshold': 0.8519254326820374, 'iso_threshold': 0.6114803972804356, 'xgb_flag': True, 'iso_flag': True, 'xgb_score': 99.94, 'isolation_score_normalized': 100.0, 'final_risk_score': 99.96}}


In [5]:
import pandas as pd

X_test = pd.read_csv("X_test.csv")

In [6]:
# Load test data
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv")

print("Engine loaded successfully!")
print(f"Number of test samples: {len(X_test)}")

Engine loaded successfully!
Number of test samples: 949


In [7]:
print("TEST 1 - FIRST-TIME USER")
result = evaluate_auction_entry(

    engine=engine,
    user_id="user001",
    starting_price=100000,
    completed_auctions=0
)

print(result)

TEST 1 - FIRST-TIME USER
{'user_id': 'user001', 'user_stage': 'new', 'risk_tier': None, 'entry_allowed': True, 'risk_score': None, 'collateral': {'starting_price': 100000, 'base_collateral': 1000.0, 'collateral_multiplier': 1.0, 'required_collateral': 1000.0}, 'details': {'user_stage': 'new', 'risk_tier': None, 'final_risk_score': None, 'xgb_probability': None, 'isolation_score': None, 'xgb_threshold': None, 'iso_threshold': None, 'xgb_flag': None, 'iso_flag': None, 'xgb_score': None, 'isolation_score_normalized': None}}


In [8]:
print("TEST 2 - SECOND AUCTION USER")

features = X_test.iloc[[0]]

result = evaluate_auction_entry(
    engine=engine,
    user_id="user001",
    starting_price=100000,
    completed_auctions=1,
    features=features
)

print(result)

TEST 2 - SECOND AUCTION USER
{'user_id': 'user001', 'user_stage': 'established', 'risk_tier': 'low', 'entry_allowed': True, 'risk_score': 13.16, 'collateral': {'starting_price': 100000, 'base_collateral': 1000.0, 'collateral_multiplier': 1.0, 'required_collateral': 1000.0}, 'details': {'user_stage': 'established', 'risk_tier': 'low', 'xgb_probability': 0.0, 'isolation_score': 0.5205, 'xgb_threshold': 0.8519254326820374, 'iso_threshold': 0.6114803972804356, 'xgb_flag': False, 'iso_flag': False, 'xgb_score': 0.0, 'isolation_score_normalized': 32.88, 'final_risk_score': 13.16}}


In [10]:
print("TEST 3 - FIRST 10 USERS")

for i in range(10):

    features = X_test.iloc[[i]]

    result = evaluate_auction_entry(
        engine=engine,
        user_id=f"user{i}",
        starting_price=100000,
        completed_auctions=1,
        features=features
    )

    print(f"\nUser {i}")
    print(f"True Label            : {y_test.iloc[i]['Class']}")
    print(f"Risk Tier             : {result['risk_tier']}")
    print(f"Risk Score            : {result['risk_score']}")
    print(f"Entry Allowed         : {result['entry_allowed']}")

    if result["collateral"]:
        print(f"Collateral Required   : {result['collateral']['required_collateral']}")
    else:
        print("Collateral Required   : Blocked")

TEST 3 - FIRST 10 USERS

User 0
True Label            : 0
Risk Tier             : low
Risk Score            : 13.16
Entry Allowed         : True
Collateral Required   : 1000.0

User 1
True Label            : 0
Risk Tier             : low
Risk Score            : 4.64
Entry Allowed         : True
Collateral Required   : 1000.0

User 2
True Label            : 0
Risk Tier             : low
Risk Score            : 5.12
Entry Allowed         : True
Collateral Required   : 1000.0

User 3
True Label            : 0
Risk Tier             : low
Risk Score            : 2.15
Entry Allowed         : True
Collateral Required   : 1000.0

User 4
True Label            : 0
Risk Tier             : low
Risk Score            : 4.79
Entry Allowed         : True
Collateral Required   : 1000.0

User 5
True Label            : 0
Risk Tier             : low
Risk Score            : 18.37
Entry Allowed         : True
Collateral Required   : 1000.0

User 6
True Label            : 0
Risk Tier             : low
Risk S

In [11]:
results = []

for i in range(20):

    features = X_test.iloc[[i]]

    result = evaluate_auction_entry(
        engine=engine,
        user_id=f"user{i}",
        starting_price=100000,
        completed_auctions=1,
        features=features
    )

    results.append({
        "Row": i,
        "Actual Class": y_test.iloc[i]["Class"],
        "Risk Tier": result["risk_tier"],
        "Risk Score": result["risk_score"],
        "Entry Allowed": result["entry_allowed"],
        "Collateral": (
            result["collateral"]["required_collateral"]
            if result["collateral"] is not None
            else 0
        ),
    })

summary = pd.DataFrame(results)

summary

,Row,Actual Class,Risk Tier,Risk Score,Entry Allowed,Collateral
0,0,0,low,13.16,True,1000.0
1,1,0,low,4.64,True,1000.0
2,2,0,low,5.12,True,1000.0
3,3,0,low,2.15,True,1000.0
4,4,0,low,4.79,True,1000.0
5,5,0,low,18.37,True,1000.0
6,6,0,low,4.20,True,1000.0
7,7,1,critical,91.46,False,0.0
8,8,0,low,7.41,True,1000.0
9,9,1,critical,89.09,False,0.0
